# AI-Powered Sales Forecasting Agent

Welcome to the **AI-Powered Sales Forecasting Agent** notebook. This system implements an agentic demand planning and revenue growth workflow using Qwen3, Meta Prophet, Scikit-Learn, and the OpenAI SDK.

### System Architecture:
```
User Question
     ↓
Qwen3 Tool Calling (via vLLM / OpenAI client)
     ↓
Python Tool Execution (Prophet, Isolation Forest, Elasticity Regression)
     ↓
Tool JSON Result
     ↓
Qwen3 Business Analysis
     ↓
Final Business Response (executive-level insights)
```

---

## 1. Setup & Ingestion
First, we will make sure the python path is configured and import the necessary libraries. If the mock sales data does not exist, we will generate it.

In [ ]:
import sys
import os
# Ensure the src directory is in the python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
from src.data.data_generator import generate_mock_sales_data

# Check if mock data is available, otherwise generate it
data_path = "../data/mock_sales.csv"
if not os.path.exists(data_path):
    print("Generating mock sales dataset...")
    generate_mock_sales_data(output_path=data_path)
else:
    print(f"Mock dataset found at: {data_path}")

Let's print a quick preview of the historical sales database to inspect the features generated.

In [ ]:
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Direct Forecasting Demonstration (with Plotly Visualizations)
Before initializing the conversational agent, let's look at how the forecasting engine works under the hood and generate an interactive Plotly visual chart.

In [ ]:
from src.forecasting.prophet_model import forecast_sales
from src.utils.helpers import plot_forecast_plotly

# Forecast sales of Product_3 at Store_1 for the next 12 weeks
results = forecast_sales(product="Product_3", store="Store_1", horizon_weeks=12, data_path=data_path)

print(results["forecast_summary"])

# Render the interactive Plotly chart
fig = plot_forecast_plotly(results["forecast_df"], title="12-Week Demand Forecast for Product_3 (Store_1)")
if fig:
    fig.show()

## 3. Initializing the Forecasting Agent
We instantiate the `SalesForecastAgent` which connects to the local vLLM endpoint. 

> **Note on local server:** If your local vLLM server is offline, the agent will gracefully fall back to **Local Offline Heuristic Mode** and compute exact mathematical results using local tools, so that this notebook remains fully runnable without API exceptions.

In [ ]:
from src.agent.agent import SalesForecastAgent

# Initialize agent
# It will attempt to connect to http://localhost:8000/v1
agent = SalesForecastAgent(base_url="http://localhost:8000/v1")

## 4. Chat Questions & Tool-Calling Loops
Now we can run user questions. Observe how the agent selects tools and processes the returns to make structured recommendations.

### Query 1: Sales Forecasting
The model will trigger the `forecast_sales` tool, analyze the growth percent, and summarize the forecast.

In [ ]:
agent.ask("Forecast Product_3 sales for next 12 weeks")

### Query 2: Promotion Impact Analysis
The model will call the `analyze_promotion_impact` tool to evaluate promotional revenue and volume uplifts.

In [ ]:
agent.ask("What was the impact of promotions for Product_3?")

### Query 3: Inventory Recommendations
The model will call `inventory_recommendation` to check which stores/products are understocked or overstocked relative to projected demand.

In [ ]:
agent.ask("Which products need inventory replenishment?")

### Query 4: Pricing Strategy (Price Elasticity)
The model will estimate price elasticity to evaluate price sensitivity for Product_5.

In [ ]:
agent.ask("Estimate price elasticity for Product_5")

### Query 5: Decomposing Forecast Drivers
The model will trigger `explain_forecast_drivers` to see how price changes, seasonal cycles, holidays, and promo campaigns split the sales volume.

In [ ]:
agent.ask("Explain the drivers of sales for Product_1 at Store_2")

### Query 6: Anomaly Detection
Scan for unexplained drops or spikes in sales volume.

In [ ]:
agent.ask("Show me any severe sales anomalies in Store_1")